In [1]:
from typing import Any, List, Callable, Union
import h5py
import pickle
from pathlib import Path
import numpy as np
from scipy.stats import pearsonr
import torch
from enformer_pytorch import Enformer
from enformer_pytorch import from_pretrained
from enformer_pytorch.finetune import HeadAdapterWrapper
from transformers import get_scheduler
from torch.utils.data import TensorDataset, DataLoader

from torch.optim import AdamW
from tqdm.auto import tqdm

/home/rajesh/projects/hackathon/SAE_Hackathon/.venv/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load data

In [2]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
device

device(type='cuda')

# Retrieve embeddings

In [8]:
import torch
import pytorch_lightning as pl
from enformer_pytorch import Enformer
from enformer_pytorch.finetune import HeadAdapterWrapper

class EnformerWithEmbeddings(pl.LightningModule):
    def __init__(self, num_tracks=25, target_layer='transformer.layers.5'):
        super().__init__()
        self.enformer = Enformer.from_pretrained('EleutherAI/enformer-official-rough')
        self.model = HeadAdapterWrapper(
            enformer=self.enformer,
            num_tracks=num_tracks,
            post_transformer_embed=False
        )
        
        # Load the pretrained weights
        # checkpoint = torch.load(model_path)
        # self.model.load_state_dict(checkpoint['model_state_dict'])
        
        self.target_layer = target_layer
        self.hook_store = {}
        self.set_hooks()
        
    def set_hooks(self):
        """Set up hooks to capture embeddings from target layer"""
        def get_activation(name):
            def hook(module, input, output):
                self.hook_store[name] = output.detach()
            return hook
        
        # Navigate to the target layer and register the hook
        layer_parts = self.target_layer.split('.')
        target = self.model.enformer
        for part in layer_parts:
            target = getattr(target, part)
        
        target.register_forward_hook(get_activation(self.target_layer))
        
    def get_embeddings(self, x):
        """Extract embeddings from the target layer and also return model predictions
        
        Returns:
            tuple: (embeddings, predictions) where embeddings are from the target layer
                  and predictions are the full model output
        """
        self.eval()
        with torch.no_grad():
            # Run a forward pass to trigger the hooks and get predictions
            predictions = self.model(x)
            # Return the captured embeddings and predictions
            return self.hook_store[self.target_layer], predictions
    
    def forward(self, x, target=None):
        preds = self.model(x)
        if target is None:
            return preds
        return self.model(seq=x, target=target)
    

# Example usage:
# model.enformer.conv_tower.5.2.to_attn_logits
# transformer.10.1.fn.4
model = EnformerWithEmbeddings(target_layer='conv_tower.5.2.to_attn_logits')
input_tensor = torch.randint(0, 5, (1, 196_608)) # for ACGTN, in that order (-1 for padding)
input_tensor.shape
outs = model.get_embeddings(input_tensor)

In [10]:
outs[0].shape, outs[1].shape

(torch.Size([1, 1536, 1536, 2]), torch.Size([1, 896, 25]))